<a href="https://colab.research.google.com/github/XiaoLiang28/Datamanagement/blob/main/ABM_Version_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import networkx as nx

N = 500
T = 100

# Government info
alpha_G = 0.2      # sensitivity to government info
G_strength = 1.0     # strength of campaign each step

p_misinfo_seed = 0.02  # Misinformation at the beginning
alpha_M = 0.3
w_negative = 0.2

# Decision-making
theta = 0.7
p_vax = 0.5

# SIR
beta = 0.05          # Infection probability
gamma = 0.1          # Recovery probability
vaccine_efficacy = 0.9


p_side_effect = 0.001
np.random.seed(0)

K = 6
P_REWIRE = 0.1

G = nx.watts_strogatz_graph(N, K, P_REWIRE)


neighbors = {i: list(G.neighbors(i)) for i in G.nodes()}




# health: 0=S, 1=I, 2=R
health = np.zeros(N, dtype=np.int8)

# vax: 0=not_vaccinated, 1=vaccinated
vax = np.zeros(N, dtype=np.int8)

# desire: scale（0~1）
desire = np.random.normal(loc=0.2, scale=0.05, size=N)
desire = np.clip(desire, 0.0, 1.0)

misinfo = (np.random.rand(N) < p_misinfo_seed)


initial_infected = max(1, N // 100)
infected_indices = np.random.choice(N, size=initial_infected, replace=False)
health[infected_indices] = 1




S_list = []
I_list = []
R_list = []
coverage_list = []

def record_stats():
    S_list.append(np.sum(health == 0))
    I_list.append(np.sum(health == 1))
    R_list.append(np.sum(health == 2))
    coverage_list.append(np.mean(vax))


def step_government_info():

    global desire

    delta = alpha_G * G_strength * (1.0 - desire)
    desire = np.clip(desire + delta, 0.0, 1.0)


def step_peer_misinformation():

    global desire

    negative_info = np.zeros(N, dtype=float)


    for i in range(N):
        if misinfo[i]:
            for nb in neighbors[i]:
                negative_info[nb] += w_negative


    for i in range(N):
        M = negative_info[i]
        if M > 0:
            d = desire[i]
            delta = -alpha_M * M * d
            desire[i] = np.clip(d + delta, 0.0, 1.0)


def step_vaccination():

    global vax
    for i in range(N):
        if vax[i] == 1:
            continue
        if desire[i] >= theta:
            if np.random.rand() < p_vax:
                vax[i] = 1


def step_infection():

    global health
    new_infected = []

    for i in range(N):
        if health[i] != 0:
            continue


        if vax[i] == 1:
            effective_beta = beta * (1 - vaccine_efficacy)
        else:
            effective_beta = beta

        prob_not_infected = 1.0
        for nb in neighbors[i]:
            if health[nb] == 1:
                prob_not_infected *= (1.0 - effective_beta)

        infection_prob = 1.0 - prob_not_infected

        if np.random.rand() < infection_prob:
            new_infected.append(i)

    for i in new_infected:
        health[i] = 1

def step_recovery():

    global health
    for i in range(N):
        if health[i] == 1 and np.random.rand() < gamma:
            health[i] = 2


def step_side_effects():

    global misinfo, desire
    for i in range(N):
        if vax[i] == 1 and (not misinfo[i]):
            if np.random.rand() < p_side_effect:
                misinfo[i] = True
                desire[i] = np.clip(desire[i] - 0.5, 0.0, 1.0)




for t in range(T):
    step_government_info()
    step_peer_misinformation()
    step_vaccination()
    step_infection()
    step_recovery()
    step_side_effects()
    record_stats()


print("Final coverage:", coverage_list[-1])
print("Final S/I/R:", S_list[-1], I_list[-1], R_list[-1])

Final coverage: 0.994
Final S/I/R: 483 0 17
